In [10]:
import os
from dotenv import load_dotenv

load_dotenv()   # reads from your .env file

# All keys are now available as environment variables
# LangChain providers pick them up automatically

True

In [11]:
from langchain.chat_models import init_chat_model

# Signature
# init_chat_model(model, model_provider=None, **kwargs)

# kwargs you can pass:
#   temperature   — controls randomness (0.0 = deterministic, 1.0 = creative)
#   max_tokens    — max length of the response
#   timeout       — seconds before the request is cancelled
#   max_retries   — how many times to retry on failure (default 6)

In [24]:
from langchain.chat_models import init_chat_model
from langchain_openrouter import ChatOpenRouter

model = ChatOpenRouter(
    model="z-ai/glm-5.1",
    temperature=0.7,
    max_tokens=500
)

response = model.invoke("What is the capital of France?")
print(response.content)

The capital of France is Paris.


In [22]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA

# Nemotron 3 Super — efficient reasoning and agentic tasks
llm = ChatNVIDIA(model="nvidia/nemotron-3-super-120b-a12b")
result = llm.invoke("list down components of Random forest")
print(result.content)

Okay,000-word answer on the components of a Random Forest, covering everything from the high‑level algorithmic idea to the low‑level details that you would implement or configure when building a Random Forest model in practice.  

I’ll break it into sections:

1. **High‑level overview** – what Random Forest is and why we use it.  
2. **Core components** – the building blocks you need to specify or understand.  
3. **Implementation details** – how each component is realized (splitting criteria, impurity measures, etc.).  
4. **Hyper‑parameters** – tunable knobs that affect the components.  
5. **Training workflow** – step‑by‑step flow from data to forest.  
6. **Prediction / inference** – how the forest aggregates tree outputs.  
7. **Optional/advanced components** – out‑of‑bag error, feature importance, parallelism, etc.  
8. **Practical checklist** – what you should verify when you build or tune a Random Forest.  

Feel free to cherry‑pick the parts you need.  

---  

## 1. High‑leve

In [26]:
from pprint import pprint

print("type(response):", type(response))
response_dict = response.model_dump() if hasattr(response, "model_dump") else response.dict()
print("top-level keys:", list(response_dict.keys()))
pprint(response_dict)


type(response): <class 'langchain_core.messages.ai.AIMessage'>
top-level keys: ['content', 'additional_kwargs', 'response_metadata', 'type', 'name', 'id', 'tool_calls', 'invalid_tool_calls', 'usage_metadata']
{'additional_kwargs': {'reasoning_content': '1.  Identify the core entity and '
                                            'the question: The user is asking '
                                            'for the capital of "France".\n'
                                            '2.  Retrieve knowledge about '
                                            'France: France is a country in '
                                            'Europe. Its capital city is '
                                            'Paris.\n'
                                            '3.  Formulate the answer: State '
                                            'clearly that the capital of '
                                            'France is Paris.',
                       'reasoning_details': [{'forma

In [27]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA

# Nemotron 3 Super — efficient reasoning and agentic tasks
llm = ChatNVIDIA(model="nvidia/nemotron-3-super-120b-a12b")

response = llm.invoke("Explain list comprehensions in Python briefly.")

# The actual text output
print("content        :", response.content)

# Message type identifier (always "ai")
print("type           :", response.type)

# Unique message ID from the provider
print("id             :", response.id)

# Optional name (usually None unless set manually)
print("name           :", response.name)

content        : List comprehensions provide a concise way to build lists by applying an expression to each item in an iterable (optionally filtering items with a condition).  
The basic syntax is:

```python
[ expression for item in iterable if condition ]
```

- **expression** – what to compute for each item (e.g., `x*2` or `item.upper()`).
- **for item in iterable** – loops over the source (list, range, string, etc.).
- **if condition** – optional filter; only items that satisfy the condition are processed.

### Examples

1. **Simple transformation**

```python
squares = [x**2 for x in range(1, 6)]
# [1, 4, 9, 16, 25]
```

2. **Filtering**

```python
evens = [x for x in range(10) if x % 2 == 0]
# [0, 2, 4, 6, 8]
```

3. **Nested loops (flattening a matrix)**

```python
matrix = [[1, 2], [3, 4], [5, 6]]
flat = [num for row in matrix for num in row]
# [1, 2, 3, 4, 5, 6]
```

4. **Conditional expression inside**

```python
labels = ['even' if x % 2 == 0 else 'odd' for x in range(5)]
# 

In [30]:
#  Token Usage (usage_metadata)

In [28]:
print("usage_metadata :", response.usage_metadata)

# Unpack each field inside usage_metadata
if response.usage_metadata:
    um = response.usage_metadata
    print("  input_tokens       :", um.get("input_tokens"))
    print("  output_tokens      :", um.get("output_tokens"))
    print("  total_tokens       :", um.get("total_tokens"))

    # These may be present for some providers (OpenAI, Anthropic)
    print("  input_token_details  :", um.get("input_token_details"))   # cache_read, audio
    print("  output_token_details :", um.get("output_token_details"))  # reasoning, audio

usage_metadata : {'input_tokens': 25, 'output_tokens': 396, 'total_tokens': 421}
  input_tokens       : 25
  output_tokens      : 396
  total_tokens       : 421
  input_token_details  : None
  output_token_details : None


In [31]:
# Provider Metadata (response_metadata)

In [32]:
import json

print("response_metadata:")
print(json.dumps(response.response_metadata, indent=2, default=str))

response_metadata:
{
  "role": "assistant",
  "content": "List comprehensions provide a concise way to build lists by applying an expression to each item in an iterable (optionally filtering items with a condition).  \nThe basic syntax is:\n\n```python\n[ expression for item in iterable if condition ]\n```\n\n- **expression** \u2013 what to compute for each item (e.g., `x*2` or `item.upper()`).\n- **for item in iterable** \u2013 loops over the source (list, range, string, etc.).\n- **if condition** \u2013 optional filter; only items that satisfy the condition are processed.\n\n### Examples\n\n1. **Simple transformation**\n\n```python\nsquares = [x**2 for x in range(1, 6)]\n# [1, 4, 9, 16, 25]\n```\n\n2. **Filtering**\n\n```python\nevens = [x for x in range(10) if x % 2 == 0]\n# [0, 2, 4, 6, 8]\n```\n\n3. **Nested loops (flattening a matrix)**\n\n```python\nmatrix = [[1, 2], [3, 4], [5, 6]]\nflat = [num for row in matrix for num in row]\n# [1, 2, 3, 4, 5, 6]\n```\n\n4. **Conditional exp

In [33]:
# Specific Keys Inside response_metadata

In [34]:
meta = response.response_metadata

# Model name used (useful when using configurable models)
print("model_name     :", meta.get("model_name"))

# Why generation stopped: "stop", "length", "tool_calls", "content_filter"
print("finish_reason  :", meta.get("finish_reason"))

# OpenAI system fingerprint (identifies model version/config)
print("system_fingerprint :", meta.get("system_fingerprint"))

# Log probabilities (only present if logprobs=True was set on the model)
print("logprobs       :", meta.get("logprobs"))

# Raw token usage dict from the provider (different from usage_metadata)
print("token_usage    :", meta.get("token_usage"))

model_name     : nvidia/nemotron-3-super-120b-a12b
finish_reason  : stop
system_fingerprint : None
logprobs       : None
token_usage    : {'prompt_tokens': 25, 'total_tokens': 421, 'completion_tokens': 396, 'prompt_tokens_details': None}


In [35]:
# Additional Kwargs

In [36]:
# Legacy field. Contains raw extra data from the provider.
# Usually empty unless tool calls are present or provider sends extra fields.
print("additional_kwargs:")
print(json.dumps(response.additional_kwargs, indent=2, default=str))

additional_kwargs:
{
  "reasoning_content": "We need to respond with brief explanation of list comprehensions in Python. Probably concise, clear. No special constraints. Provide example.\n\n",
  "reasoning": "We need to respond with brief explanation of list comprehensions in Python. Probably concise, clear. No special constraints. Provide example.\n\n",
  "_reasoning_api_fields": [
    "reasoning_content",
    "reasoning"
  ]
}


In [37]:
# Tool Calls (when the model called a tool)

In [40]:
from langchain.tools import tool


@tool
def get_weather_data(city: str) -> dict:
    """Get structured weather data for a city."""
    return {
        "city": city,
        "temperature_c": 22,
        "conditions": "sunny",
    }

In [42]:
from langchain.tools import tool
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.messages import HumanMessage

@tool
def get_weather_data(city: str) -> dict:
    """Get structured weather data for a city.
    Use this tool to fetch current weather information for a specified city."""
    return {
        "city": city,
        "temperature_c": 22,
        "conditions": "sunny",
    }

# Initialize your language model
llm = ChatNVIDIA(model="nvidia/nemotron-3-super-120b-a12b")

# Bind the tool to the LLM
llm_with_tools = llm.bind_tools([get_weather_data])

# Invoke the model with a message that requires tool usage
response = llm_with_tools.invoke([HumanMessage(content="What's the weather like in London?")])  # type: ignore

# Print the response, which will include tool calls if the model decides to use them
print(response.content)

In [43]:
# List of tool calls the model is requesting to make
print("tool_calls:", response.tool_calls) #type: ignore

# Loop through each tool call
for tc in response.tool_calls: #type: ignore
    print("  name :", tc["name"])
    print("  args :", tc["args"])
    print("  id   :", tc["id"])
    print("  type :", tc["type"])  # always "tool_call" #type:ignore

# Tool calls that had parsing errors
print("invalid_tool_calls:", response.invalid_tool_calls) #type: ignore

tool_calls: [{'name': 'get_weather_data', 'args': {'city': 'London'}, 'id': 'chatcmpl-tool-ab6e6ea88753b960', 'type': 'tool_call'}]
  name : get_weather_data
  args : {'city': 'London'}
  id   : chatcmpl-tool-ab6e6ea88753b960
  type : tool_call
invalid_tool_calls: []


In [44]:
# Content Blocks
# content_blocks gives structured typed access to the response content.
# Especially useful when the model returns reasoning, images, or multiple parts.

In [48]:
# content_blocks is already a list (usually list[dict]), so don't call __dict__()
print("type:", type(response.content_blocks))
print("content_blocks:", response.content_blocks)

for i, block in enumerate(response.content_blocks):
    print(f"\nBlock {i}:")
    print("  type:", block.get("type") if isinstance(block, dict) else type(block))
    print("  data:", block)

type: <class 'list'>
content_blocks: [{'type': 'reasoning', 'reasoning': 'I need to get weather data for London using the get_weather_data tool. Let me call that function.\n'}, {'type': 'tool_call', 'id': 'chatcmpl-tool-ab6e6ea88753b960', 'name': 'get_weather_data', 'args': {'city': 'London'}}]

Block 0:
  type: reasoning
  data: {'type': 'reasoning', 'reasoning': 'I need to get weather data for London using the get_weather_data tool. Let me call that function.\n'}

Block 1:
  type: tool_call
  data: {'type': 'tool_call', 'id': 'chatcmpl-tool-ab6e6ea88753b960', 'name': 'get_weather_data', 'args': {'city': 'London'}}


In [45]:
print("content_blocks:", response.content_blocks)

for block in response.content_blocks:
    print("  block type: ", block['type'])

    if block['type'] == "text":
        print("  text:", block["text"])

    elif block['type'] == "reasoning":
        print("  reasoning:", block.get("reasoning"))

    elif block['type'] == "tool_use":
        print("  tool_name :", block.get("name"))
        print("  tool_input:", block.get("input"))

    elif block["type"] == "image":
        print("  image mime_type:", block.get("mime_type"))
        print("  image base64 (first 40 chars):", str(block.get("base64", ""))[:40])                


content_blocks: [{'type': 'reasoning', 'reasoning': 'I need to get weather data for London using the get_weather_data tool. Let me call that function.\n'}, {'type': 'tool_call', 'id': 'chatcmpl-tool-ab6e6ea88753b960', 'name': 'get_weather_data', 'args': {'city': 'London'}}]


In [49]:
print("content_blocks:", response.content_blocks)

for block in response.content_blocks:
    print("  block type:", block["type"])

    if block["type"] == "text":
        print("  text:", block["text"])

    elif block["type"] == "reasoning":
        print("  reasoning:", block.get("reasoning"))

    elif block["type"] == "tool_use":
        print("  tool_name :", block.get("name"))
        print("  tool_input:", block.get("input"))

    elif block["type"] == "image":
        print("  image mime_type:", block.get("mime_type"))
        print("  image base64 (first 40 chars):", str(block.get("base64", ""))[:40])

content_blocks: [{'type': 'reasoning', 'reasoning': 'I need to get weather data for London using the get_weather_data tool. Let me call that function.\n'}, {'type': 'tool_call', 'id': 'chatcmpl-tool-ab6e6ea88753b960', 'name': 'get_weather_data', 'args': {'city': 'London'}}]
  block type: reasoning
  reasoning: I need to get weather data for London using the get_weather_data tool. Let me call that function.

  block type: tool_call


In [50]:
# Serialization Fields (Internal / Debug)

In [51]:
# Attributes included in LangChain serialization
print("lc_attributes  :", response.lc_attributes)

# Class identifier path used for serialization/deserialization
print("lc_id          :", response.lc_id())

# Whether this class supports LangChain serialization
print("is_lc_serializable:", response.is_lc_serializable())

# The module namespace of this class
print("lc_namespace   :", response.get_lc_namespace())

lc_attributes  : {'tool_calls': [{'name': 'get_weather_data', 'args': {'city': 'London'}, 'id': 'chatcmpl-tool-ab6e6ea88753b960', 'type': 'tool_call'}], 'invalid_tool_calls': []}
lc_id          : ['langchain', 'schema', 'messages', 'AIMessage']
is_lc_serializable: True
lc_namespace   : ['langchain', 'schema', 'messages']


In [52]:
# Print Everything at Once

In [56]:
import json
from langchain.chat_models import init_chat_model

model = init_chat_model("openai/gpt-oss-120b", model_provider="groq")
response = model.invoke("What is Python?")

print("———" * 30)
print("FULL AIMessage DEBUG")
print("———" * 30)

print("\n——— BASIC FIELDS ———")
print("type           :", response.type)
print("id             :", response.id)
print("name           :", response.name)
print("content        :", response.content)

print("\n——— USAGE METADATA (standardized) ———")
print(json.dumps(response.usage_metadata, indent=2, default=str))

print("\n——— RESPONSE METADATA (raw provider data) ———")
print(json.dumps(response.response_metadata, indent=2, default=str))

print("\n——— ADDITIONAL KWARGS ———")
print(json.dumps(response.additional_kwargs, indent=2, default=str))

print("\n——— TOOL CALLS ———")
print(response.tool_calls)

print("\n——— INVALID TOOL CALLS ———")
print(response.invalid_tool_calls)

print("\n——— CONTENT BLOCKS ———")
for i, block in enumerate(response.content_blocks):
    print(f"  block[{i}]:", block)

print("\n——— SERIALIZATION INFO ———")
print("lc_attributes      :", response.lc_attributes)
print("is_lc_serializable :", response.is_lc_serializable())
print("lc_namespace       :", response.get_lc_namespace())
print("lc_id              :", response.lc_id())

print("———" * 30)

——————————————————————————————————————————————————————————————————————————————————————————
FULL AIMessage DEBUG
——————————————————————————————————————————————————————————————————————————————————————————

——— BASIC FIELDS ———
type           : ai
id             : lc_run--019d802b-7032-7e30-8a94-d7958340ca94-0
name           : None
content        : **Python** is a high‑level, interpreted programming language known for its readability, simplicity, and versatility. Here’s a quick overview:

---

## 1. Core Characteristics
| Feature | What It Means |
|---------|---------------|
| **Interpreted** | Code is executed line‑by‑line by the Python interpreter; no separate compilation step is required. |
| **Dynamically typed** | Variable types are inferred at runtime, so you don’t need to declare them explicitly. |
| **Indentation‑based syntax** | Blocks of code are defined by indentation (whitespace) rather than braces or keywords, which encourages clean, readable code. |
| **Multi‑paradigm** | Su

In [58]:
# Returns a formatted string for display
print(response.pretty_repr())

================================== Ai Message ==================================

**Python** is a high‑level, interpreted programming language known for its readability, simplicity, and versatility. Here’s a quick overview:

---

## 1. Core Characteristics
| Feature | What It Means |
|---------|---------------|
| **Interpreted** | Code is executed line‑by‑line by the Python interpreter; no separate compilation step is required. |
| **Dynamically typed** | Variable types are inferred at runtime, so you don’t need to declare them explicitly. |
| **Indentation‑based syntax** | Blocks of code are defined by indentation (whitespace) rather than braces or keywords, which encourages clean, readable code. |
| **Multi‑paradigm** | Supports procedural, object‑oriented, functional, and even some aspects of imperative programming. |
| **Extensive standard library** | “Batteries‑included” modules for tasks such as file I/O, networking, data serialization, testing, and more. |
| **Cross‑platform** |

In [59]:
# Directly prints to stdout — same result but no return value
response.pretty_print()

================================== Ai Message ==================================

**Python** is a high‑level, interpreted programming language known for its readability, simplicity, and versatility. Here’s a quick overview:

---

## 1. Core Characteristics
| Feature | What It Means |
|---------|---------------|
| **Interpreted** | Code is executed line‑by‑line by the Python interpreter; no separate compilation step is required. |
| **Dynamically typed** | Variable types are inferred at runtime, so you don’t need to declare them explicitly. |
| **Indentation‑based syntax** | Blocks of code are defined by indentation (whitespace) rather than braces or keywords, which encourages clean, readable code. |
| **Multi‑paradigm** | Supports procedural, object‑oriented, functional, and even some aspects of imperative programming. |
| **Extensive standard library** | “Batteries‑included” modules for tasks such as file I/O, networking, data serialization, testing, and more. |
| **Cross‑platform** |

In [60]:
#  Convert to Plain Python Dict

# Converts the entire AIMessage into a plain Python dict
data = response.model_dump()
print(type(data))   # <class 'dict'>
print(data.keys())
# dict_keys(['content', 'additional_kwargs', 'response_metadata', 'type',
#            'name', 'id', 'tool_calls', 'invalid_tool_calls', 'usage_metadata'])

# Access fields from the dict
print(data["content"])
print(data["usage_metadata"])
print(data["response_metadata"])

<class 'dict'>
dict_keys(['content', 'additional_kwargs', 'response_metadata', 'type', 'name', 'id', 'tool_calls', 'invalid_tool_calls', 'usage_metadata'])
**Python** is a high‑level, interpreted programming language known for its readability, simplicity, and versatility. Here’s a quick overview:

---

## 1. Core Characteristics
| Feature | What It Means |
|---------|---------------|
| **Interpreted** | Code is executed line‑by‑line by the Python interpreter; no separate compilation step is required. |
| **Dynamically typed** | Variable types are inferred at runtime, so you don’t need to declare them explicitly. |
| **Indentation‑based syntax** | Blocks of code are defined by indentation (whitespace) rather than braces or keywords, which encourages clean, readable code. |
| **Multi‑paradigm** | Supports procedural, object‑oriented, functional, and even some aspects of imperative programming. |
| **Extensive standard library** | “Batteries‑included” modules for tasks such as file I/O, n

In [61]:
# to_json — Serialize to LangChain JSON Format
import json

# Serialize to a LangChain-compatible JSON structure
serialized = response.to_json()
print(type(serialized))   # <class 'dict'>

# Pretty print
print(json.dumps(serialized, indent=2, default=str))

<class 'dict'>
{
  "lc": 1,
  "type": "constructor",
  "id": [
    "langchain",
    "schema",
    "messages",
    "AIMessage"
  ],
  "kwargs": {
    "content": "**Python** is a high\u2011level, interpreted programming language known for its readability, simplicity, and versatility. Here\u2019s a quick overview:\n\n---\n\n## 1. Core Characteristics\n| Feature | What It Means |\n|---------|---------------|\n| **Interpreted** | Code is executed line\u2011by\u2011line by the Python interpreter; no separate compilation step is required. |\n| **Dynamically typed** | Variable types are inferred at runtime, so you don\u2019t need to declare them explicitly. |\n| **Indentation\u2011based syntax** | Blocks of code are defined by indentation (whitespace) rather than braces or keywords, which encourages clean, readable code. |\n| **Multi\u2011paradigm** | Supports procedural, object\u2011oriented, functional, and even some aspects of imperative programming. |\n| **Extensive standard library** | \u

In [62]:
# dumps / dumpd — LangChain Serialization Utilities

from langchain_core.load import dumps, dumpd

# dumps: serialize to JSON string
json_string = dumps(response, pretty=True)
print(json_string[:300])

# dumpd: serialize to a plain Python dict
dict_form = dumpd(response)
print(dict_form["kwargs"]["content"])

{
  "lc": 1,
  "type": "constructor",
  "id": [
    "langchain",
    "schema",
    "messages",
    "AIMessage"
  ],
  "kwargs": {
    "content": "**Python** is a high\u2011level, interpreted programming language known for its readability, simplicity, and versatility. Here\u2019s a quick overview:\n\
**Python** is a high‑level, interpreted programming language known for its readability, simplicity, and versatility. Here’s a quick overview:

---

## 1. Core Characteristics
| Feature | What It Means |
|---------|---------------|
| **Interpreted** | Code is executed line‑by‑line by the Python interpreter; no separate compilation step is required. |
| **Dynamically typed** | Variable types are inferred at runtime, so you don’t need to declare them explicitly. |
| **Indentation‑based syntax** | Blocks of code are defined by indentation (whitespace) rather than braces or keywords, which encourages clean, readable code. |
| **Multi‑paradigm** | Supports procedural, object‑oriented, functional,

In [63]:
# Printing finish_reason — Why Did the Model Stop?

finish_reason = response.response_metadata.get("finish_reason")
print("finish_reason:", finish_reason)

# Possible values:
# "stop"           — model finished naturally
# "length"         — hit max_tokens limit
# "tool_calls"     — model is calling a tool (not done yet)
# "content_filter" — output was filtered

finish_reason: stop


In [64]:
# Using dir() to Discover All Attributes at Runtime

# Print all non-dunder attributes and methods on an AIMessage
attrs = [a for a in dir(response) if not a.startswith("__")]
for a in attrs:
    print(a)

_abc_impl
_backwards_compat_tool_calls
_calculate_keys
_copy_and_set_values
_get_value
_iter
_setattr_handler
additional_kwargs
construct
content
content_blocks
copy
dict
from_orm
get_lc_namespace
id
invalid_tool_calls
is_lc_serializable
json
lc_attributes
lc_id
lc_secrets
model_computed_fields
model_config
model_construct
model_copy
model_dump
model_dump_json
model_extra
model_fields
model_fields_set
model_json_schema
model_parametrized_name
model_post_init
model_rebuild
model_validate
model_validate_json
model_validate_strings
name
parse_file
parse_obj
parse_raw
pretty_print
pretty_repr
response_metadata
schema
schema_json
text
to_json
to_json_not_implemented
tool_calls
type
update_forward_refs
usage_metadata
validate


In [66]:
# Accessing text Property (Shortcut)
# .text is a shortcut that returns the string content
# It is a TextAccessor object that behaves like a string
print(response.text)

# Equivalent to:
# print(response.content)

**Python** is a high‑level, interpreted programming language known for its readability, simplicity, and versatility. Here’s a quick overview:

---

## 1. Core Characteristics
| Feature | What It Means |
|---------|---------------|
| **Interpreted** | Code is executed line‑by‑line by the Python interpreter; no separate compilation step is required. |
| **Dynamically typed** | Variable types are inferred at runtime, so you don’t need to declare them explicitly. |
| **Indentation‑based syntax** | Blocks of code are defined by indentation (whitespace) rather than braces or keywords, which encourages clean, readable code. |
| **Multi‑paradigm** | Supports procedural, object‑oriented, functional, and even some aspects of imperative programming. |
| **Extensive standard library** | “Batteries‑included” modules for tasks such as file I/O, networking, data serialization, testing, and more. |
| **Cross‑platform** | Runs on Windows, macOS, Linux, and many other operating systems, plus embedded de

In [67]:
# Checking Token Counts — Quick Helpers

In [68]:
# Quick function to print a token summary after any invoke call
def token_summary(response):
    um = response.usage_metadata
    if um:
        print(f"Input tokens  : {um.get('input_tokens', 'N/A')}")
        print(f"Output tokens : {um.get('output_tokens', 'N/A')}")
        print(f"Total tokens  : {um.get('total_tokens', 'N/A')}")
        
        # Cache info (Anthropic)
        details = um.get("input_token_details", {})
        if details.get("cache_read"):
            print(f"Cache read    : {details['cache_read']}")
        if details.get("cache_creation"):
            print(f"Cache created : {details['cache_creation']}")
        
        # Reasoning tokens (OpenAI o-series)
        out_details = um.get("output_token_details", {})
        if out_details.get("reasoning"):
            print(f"Reasoning tokens : {out_details['reasoning']}")
    else:
        print("No usage metadata returned by this provider.")

token_summary(response)

Input tokens  : 75
Output tokens : 1032
Total tokens  : 1107
Reasoning tokens : 53


In [69]:
# Extracting Model Name from the Response

meta = response.response_metadata

# OpenAI style
model_used = meta.get("model_name")

# Anthropic style
if not model_used:
    model_used = meta.get("model")

# Groq style
if not model_used:
    model_used = meta.get("model_name")

print("Model actually used:", model_used)

Model actually used: openai/gpt-oss-120b
